# **Camada Silver**

- Importando a base de dados resultante da camada bronze

In [ ]:
df = pd.read_csv('../Data/Raw/tabela_bronze.csv')

## **Limpeza dos Dados**

In [ ]:
# Limpar espaços nos textos e transformar campos vazios em nulos
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].str.strip()

df = df.replace(['', 'null', 'NULL', 'unknown'], pd.NA)
df['customerID'] = df['customerID'].str.upper()

# ID válido: quatro números, hífen e cinco letras (exemplo: 4578-PHJYZ)
id_valido = df['customerID'].str.fullmatch(r'[0-9]{4}-[A-Z]{5}', na=False)

df_linhas_invalidas = df[~id_valido].copy()
df = df[id_valido].copy()

# Remover somente linhas totalmente iguais
df = df.drop_duplicates().reset_index(drop=True)
print('Clientes:', len(df))
print('Linhas separadas para conferir:', len(df_linhas_invalidas))

## **Tradução dos Dados**

In [ ]:
# Traduções usadas em mais de uma coluna
sim_nao = {
    'yes': 'Sim',
    'no': 'Não'
}

servico_internet = {
    'yes': 'Sim',
    'no': 'Não',
    'no internet service': 'Sem serviço de internet'
}

# Tradução de cada coluna
traducao_valores = {
    'gender': {
        'male': 'Masculino',
        'female': 'Feminino'
    },
    'Partner': sim_nao,
    'Dependents': sim_nao,
    'PhoneService': sim_nao,
    'PaperlessBilling': sim_nao,
    'Churn': sim_nao,

    'MultipleLines': {
        'yes': 'Sim',
        'no': 'Não',
        'no phone service': 'Sem serviço de telefonia'
    },
    'InternetService': {
        'dsl': 'DSL',
        'fiber optic': 'Fibra óptica',
        'no': 'Sem serviço de internet'
    },
    'OnlineSecurity': servico_internet,
    'OnlineBackup': servico_internet,
    'DeviceProtection': servico_internet,
    'TechSupport': servico_internet,
    'StreamingTV': servico_internet,
    'StreamingMovies': servico_internet,

    'Contract': {
        'month-to-month': 'Mensal',
        'one year': 'Um ano',
        'two year': 'Dois anos'
    },
    'PaymentMethod': {
        'electronic check': 'Cheque eletrônico',
        'mailed check': 'Cheque pelo correio',
        'bank transfer (automatic)': 'Transferência bancária automática',
        'credit card (automatic)': 'Cartão de crédito automático'
    }
}

# Colocar os textos em minúsculas e aplicar as traduções
for coluna, mapa in traducao_valores.items():
    df[coluna] = df[coluna].str.lower().replace(mapa)

# Conferir o resultado
df[['gender', 'InternetService', 'Contract', 'PaymentMethod', 'Churn']].head()


## **Tradução das Colunas**

In [ ]:
# Traduzir as colunas de acordo com a descrição da sua base
traducao_colunas = {
    'customerID': 'ID_Cliente', 
    'gender': 'Genero',
    'SeniorCitizen': 'Idoso',
    'Partner': 'Conjuge', 
    'Dependents': 'Dependentes', 
    'tenure': 'Tempo_Assinatura',
    'PhoneService': 'Servico_Telefonia', 
    'MultipleLines': 'Varias_Linhas',
    'InternetService': 'Servico_Internet', 
    'OnlineSecurity': 'Seguranca_Online',
    'OnlineBackup': 'Backup_Online', 
    'DeviceProtection': 'Protecao_Dispositivos',
    'TechSupport': 'Suporte_Tecnico', 
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filmes', 
    'Contract': 'Contrato',
    'PaperlessBilling': 'Faturamento_Eletronico', 
    'PaymentMethod': 'Forma_Pagamento',
    'MonthlyCharges': 'Valores_Mensais', 
    'TotalCharges': 'Valores_Totais',
    'Churn': 'Status_Cancelamento', 
    'CustomerFeedback': 'Feedback_Cliente',
    'MonthlyIncome': 'Renda_Mensal'
}
df = df.rename(columns=traducao_colunas)

- Visualizando o resultado

In [ ]:
df.head()

## **Tipagem dos dados**

In [ ]:
# Números: tempo em meses, cobranças e renda
colunas_numericas = ['Tempo_Assinatura', 'Idoso', 'Valores_Mensais', 'Valores_Totais', 'Renda_Mensal']
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='raise')

df = df.astype({
    'Tempo_Assinatura': 'Int64',
    'Valores_Mensais': 'Float64',
    'Valores_Totais': 'Float64',
    'Renda_Mensal': 'Float64'
})

df['Idoso'] = df['Idoso'].astype('boolean')

colunas_booleanas = ['Conjuge', 'Dependentes', 'Servico_Telefonia',
                     'Faturamento_Eletronico', 'Status_Cancelamento']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].replace({'Sim': True, 'Não': False}).astype('boolean')

colunas_categoricas = ['Genero', 'Servico_Internet', 'Varias_Linhas',
                      'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivos',
                      'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes',
                      'Contrato', 'Forma_Pagamento']
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype('category')

- Olhando a nova tipagem dos dados

In [ ]:
df.info()

- Visualizando a tabela de números nulos

In [ ]:
df_silver = df.copy()

tabela_nulos = pd.DataFrame({
    'Total Nulos': df_silver.isnull().sum(),
    'Porcentagem (%)': (df_silver.isnull().mean() * 100).round(2)
})
tabela_nulos

- Conferindo resultados

In [ ]:
df_silver.head(10)